# Make sure it's all in there

In [ ]:
from pathlib import Path
from sklearn.metrics import adjusted_rand_score
import scanpy as sc
import statistics as stats
import pandas as pd
import os
import numpy as np
from sklearn.metrics.cluster import adjusted_mutual_info_score
import matplotlib.pyplot as plt
#import scvi
import warnings
warnings.filterwarnings('ignore')

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (3,3)

In [ ]:
base_dir = '/home/workspace/private/projects/kim/drg/panel'

panel_path = os.path.join(base_dir, 'gene-lists/drafts/2025-07-09_gene-list-draft.csv') # optimized
source_path = os.path.join(base_dir, 'gene-lists/output/2025-07-08_drg-draft-panel-filtered.csv')

output_path = os.path.join(base_dir, 'gene-lists/drafts/2025-07-09_panel-draft.csv')

In [ ]:
# read panel (de-duplicated is 479 genes)
panel = pd.read_csv(panel_path)
panel

In [ ]:
gene_source_matrix = pd.read_csv(source_path)
gene_source_matrix

# merge

In [ ]:
merged_df = panel.merge(gene_source_matrix, on='Gene', how='left')
merged_df.tail(50)

In [ ]:
# For the neuron DEG genes at the end
false_columns = [
    'zhen_list',
    'spectral_panel',
    'GSE139088_supervised',
    'GSE139088_unsupervised',
    'GSE254789_supervised',
    'GSE254789_unsupervised'
]

# Set 'add_on' to True for rows 448 to 472
merged_df.loc[448:472, 'add_on'] = True

# Set the listed columns to False for the same rows
merged_df.loc[448:472, false_columns] = False

In [ ]:
merged_df

In [ ]:
#merged_df.to_csv(output_path, index = False)

In [ ]:
missing_genes = gene_source_matrix[~gene_source_matrix['Gene'].isin(merged_df['Gene'])]
print(missing_genes['Gene'])

# UMAPs

## adata

In [ ]:
# Directories
base_dir = Path("/home/workspace/private/projects/kim/drg")
data_dir = base_dir / "data" / "scrna-seq"
input_dir = data_dir / "h5ad" / "04_clustered"

In [ ]:
adata = sc.read_h5ad(input_dir / "GSE254789-nonneurons.h5ad")

In [ ]:
sc.pl.umap(adata, color = 'cell_type', title = '', frameon = False)

In [ ]:
for gene in merged_df['Gene']:
    if gene in adata.var_names:  # Make sure gene is in the dataset
        sc.pl.umap(adata, color=gene, show=True, frameon = False)
    else:
        print(f"Gene '{gene}' not found in adata.var_names, skipping.")

In [ ]:
adata

In [ ]:
import numpy as np

# Total counts per cell (already stored in adata)
median_umi = np.median(adata.obs['n_counts'] if 'n_counts' in adata.obs else adata.X.sum(axis=1).A1)
print(f"Median transcript count per cell: {median_umi:.0f}")

In [ ]:
# Count detected genes per cell (nonzero genes)
nonzero_gene_counts = (adata.X > 0).sum(axis=1)
median_genes = np.median(np.array(nonzero_gene_counts).flatten())

print(f"Median number of detected genes per cell: {median_genes:.0f}")

## bdata

In [ ]:
bdata = sc.read_h5ad(input_dir / "GSE139088-scvi-leiden.h5ad")

In [ ]:
sc.pl.umap(bdata, color = 'original_annotation', title = 'Author Annotations', frameon = False)

In [ ]:
for gene in merged_df['Gene']:
    if gene in bdata.var_names:  # Make sure gene is in the dataset
        sc.pl.umap(bdata, color=gene, show=True, frameon = False)
    else:
        print(f"Gene '{gene}' not found in adata.var_names, skipping.")

In [ ]:
bdata

In [ ]:
import numpy as np

# Safely compute median UMI count per cell
if 'n_counts' in bdata.obs:
    median_umi = np.median(bdata.obs['n_counts'])
else:
    median_umi = np.median(np.array(bdata.X.sum(axis=1)).flatten())

print(f"Median transcript count per cell: {median_umi:.0f}")

In [ ]:
# Count detected genes per cell (nonzero genes)
nonzero_gene_counts = (bdata.X > 0).sum(axis=1)
median_genes = np.median(np.array(nonzero_gene_counts).flatten())

print(f"Median number of detected genes per cell: {median_genes:.0f}")

In [ ]:
sc.pl.umap(bdata, color = ['Mbp', 'Plp1', 'Fabp7'], frameon = False)

In [ ]:
sc.pl.umap(adata, color = ['leiden'], frameon = False, legend_loc = 'on data', title = "Leiden Clusters (Non-neurons)", legend_fontsize = 9, legend_fontoutline=1)

In [ ]:
import seaborn as sns
palette = sns.color_palette("Set2", 29)

In [ ]:
sc.pl.umap(bdata, color = ['leiden'], frameon = False, legend_loc = 'on data', title = "Leiden Clusters (Neurons)", legend_fontsize = 6, legend_fontoutline=1)

In [ ]:
bdata.obs.leiden.value_counts()